# Basics of Text Generation with LLMs

<sup>This notebook is a part of the Natural Language Processing class at the University of Ljubljana, Faculty of Computer and Information Science.</sup>

---

Large Language Models generate text **one token at a time**, guided by a learned probability distribution over the vocabulary. The `.generate()` method exposes a rich set of parameters that shape how this process unfolds — controlling output length, randomness, quality, and diversity.

We cover:
1. How LLMs generate text (autoregressive decoding)
2. Loading a quantised instruction-tuned LLM (4-bit / BitsAndBytes)
3. `max_new_tokens` — controlling output length
4. `do_sample` — greedy vs. stochastic decoding
5. `temperature` — sharpness of the probability distribution
6. `top_k` — vocabulary truncation to the top-*k* tokens
7. `top_p` — nucleus (dynamic vocabulary) sampling
8. `repetition_penalty` — penalising repeated tokens
9. `num_beams` — beam search
10. `num_return_sequences` — generating multiple candidates
11. `no_repeat_ngram_size` — hard n-gram repetition penalty

## 1  Setup

We need the following libraries:

| Library | Purpose |
|---|---|
| `transformers` | Model loading and `.generate()` API |
| `bitsandbytes` | 4-bit quantisation (reduces GPU memory ~4×) |
| `accelerate` | Device placement and mixed-precision support |

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate

In [18]:
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

## 2  How LLMs Generate Text

Understanding the mechanics behind text generation makes every tuning parameter intuitive.

### Autoregressive Generation

An LLM is a function that maps a token sequence to a **probability distribution** over the next token:

$$P(w_t \mid w_1, w_2, \dots, w_{t-1})$$

Generation is **autoregressive**: the model picks one token, appends it to the input, and repeats — until a stop condition (`max_new_tokens` or an end-of-sequence token) is reached.

```
Token 1 → Token 2 → Token 3 → … → <EOS>
```

Because each step conditions on *all* previously generated tokens, early choices propagate forward through the entire sequence.

### The Probability Distribution

At each step the model produces a **logit vector** of size *V* (vocabulary size — typically 32 000–150 000 tokens). A softmax converts logits to probabilities:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Sampling parameters (`temperature`, `top_k`, `top_p`) **reshape this distribution** before a token is drawn. Parameters such as `repetition_penalty` modify the logits directly, before the softmax.

### Why Sampling Parameters Matter

With a **peaked** distribution one token dominates — the model is deterministic but can get stuck in loops.  
With a **flat** distribution many tokens are equally likely — outputs are diverse but often incoherent.

The sampling parameters let us tune this trade-off:

```
Low temperature / small k/p  →  conservative, focused, repetitive
High temperature / large k/p →  creative, diverse, potentially incoherent
```

## 3  Loading the Model

We use **Qwen/Qwen2.5-1.5B-Instruct** — a compact (1.5 B parameters), instruction-tuned model that runs comfortably on a single GPU after 4-bit quantisation.

### Why 4-bit quantisation?

| Precision | Bytes / param | Memory for 1.5 B model |
|---|---|---|
| float32 | 4 | ~6 GB |
| bfloat16 | 2 | ~3 GB |
| nf4 (4-bit) | 0.5 | **~0.75 GB** |

**NF4** (NormalFloat 4-bit) spaces its 16 quantisation levels to match the cumulative distribution of normally-distributed weights, minimising quantisation error (Dettmers et al., 2023). Computations are still performed in `bfloat16` — the weights are upcast on the fly.

In [19]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # NormalFloat4 — optimal for bell-curve weight distributions
    bnb_4bit_use_double_quant=True,         # Quantise the quantisation constants too (~0.37 bits saved/param)
    bnb_4bit_compute_dtype=torch.bfloat16,  # Upcast to bfloat16 for matrix multiplications
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",   # Distribute layers across available GPUs automatically
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model.eval()  # Disable dropout for deterministic outputs

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

In [20]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def generate(prompt, system="You are a helpful assistant.", **kwargs):
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]
    result = pipe(messages, **kwargs)
    return result[0]["generated_text"][-1]["content"]

## 4  `max_new_tokens` — Controlling Output Length

`max_new_tokens` caps the **number of tokens generated in a single call**. Generation stops earlier if an end-of-sequence token appears first.

**Why it matters:** Too small and the response is truncated mid-thought; too large and generation is slow and may drift off-topic. Setting a value appropriate for your task is the cheapest optimisation available.

In [21]:
PROMPT = "What does the attention mechanism in a transformer do? Answer in 2-3 sentences."

for n in [20, 80, 250]:
    print(f"=== max_new_tokens={n} ===")
    print(generate(PROMPT, do_sample=False, max_new_tokens=n))
    print()

=== max_new_tokens=20 ===
The attention mechanism in a transformer allows it to focus on different parts of an input sequence during its processing

=== max_new_tokens=80 ===
The attention mechanism in a transformer allows it to focus on different parts of an input sequence during its processing, similar to how humans can selectively attend to certain aspects of their environment. This is achieved by introducing a self-attention layer that computes weighted scores between elements of the input and output sequences based on their similarity or relevance. These scores determine which parts of the input contribute most to the final output, enabling

=== max_new_tokens=250 ===
The attention mechanism in a transformer allows it to focus on different parts of an input sequence during its processing, similar to how humans can selectively attend to certain aspects of their environment. This is achieved by introducing a self-attention layer that computes weighted scores between elements of the 

With `max_new_tokens=20` the explanation is cut off mid-sentence. At `80` we get a concise but almost complete answer. At `250` the model has room to elaborate with examples and details.

## 5  `do_sample` — Greedy vs. Stochastic Decoding

| Mode | Mechanism | Character |
|---|---|---|
| `do_sample=False` *(default)* | Always pick the highest-probability token | Deterministic — identical output every run |
| `do_sample=True` | Draw from the probability distribution | Stochastic — different output each run |

**Greedy decoding** is reproducible and predictable — useful for factual or structured tasks.  
**Sampling** introduces diversity — essential for creative, conversational, or data-augmentation tasks.

In [22]:
FACTUAL_PROMPT = "What is the boiling point of water in Celsius, and why does pressure affect it?"

print("=== Greedy (do_sample=False) — run twice: output is identical ===")
print("RUN 1: ", generate(FACTUAL_PROMPT, do_sample=False, max_new_tokens=80))
print()
print("RUN 2: ", generate(FACTUAL_PROMPT, do_sample=False, max_new_tokens=80))

print("\n=== Sampling (do_sample=True) — run twice: output varies ===")
torch.manual_seed(1)
print("RUN 1: ", generate(FACTUAL_PROMPT, do_sample=True, max_new_tokens=80, temperature=0.8))
print()
torch.manual_seed(2)
print("RUN 2: ", generate(FACTUAL_PROMPT, do_sample=True, max_new_tokens=80, temperature=0.8))

=== Greedy (do_sample=False) — run twice: output is identical ===
RUN 1:  The boiling point of water at standard atmospheric pressure (1 atmosphere or 760 mmHg) is 100 degrees Celsius. However, if you increase the pressure on water, its boiling point increases as well. For example, at a pressure of 2 atmospheres (2 x 760 mmHg = 1520 mmHg), the boiling point

RUN 2:  The boiling point of water at standard atmospheric pressure (1 atmosphere or 760 mmHg) is 100 degrees Celsius. However, if you increase the pressure on water, its boiling point increases as well. For example, at a pressure of 2 atmospheres (2 x 760 mmHg = 1520 mmHg), the boiling point

=== Sampling (do_sample=True) — run twice: output varies ===
RUN 1:  The boiling point of water at standard atmospheric pressure (1 atmosphere or 760 mmHg) is 100°C. However, if you change the pressure around the water, its boiling point will also change. At higher pressures, the liquid phase freezes into a solid called ice instead of evapora

The two greedy outputs are word-for-word identical. The sampled outputs carry the same facts but differ in phrasing, sentence order, or which aspect they emphasise first.

## 6  `temperature` — Sharpness of the Distribution

Temperature *T* is applied by **dividing logits before the softmax**:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

| Temperature | Effect | Output character |
|---|---|---|
| *T* < 1 | Distribution sharpened — the most likely token dominates | Conservative, repetitive |
| *T* = 1 | Distribution unchanged — model's native probabilities | Balanced |
| *T* > 1 | Distribution flattened — probabilities equalised | Creative, unpredictable |

> Setting `temperature → 0` approaches greedy decoding. Very high values (> 2) typically produce incoherent text.

Requires `do_sample=True` — temperature has no effect in greedy mode.

In [23]:
CREATIVE_PROMPT = "Begin a short story about a robot who discovers an old library."

for temp in [0.1, 0.7, 1.5]:
    torch.manual_seed(42)
    print(f"=== temperature={temp} ===")
    print(generate(CREATIVE_PROMPT, do_sample=True, max_new_tokens=80, temperature=temp))
    print()

=== temperature=0.1 ===
In the quiet hum of its own circuits, the robot had been programmed to explore the vast expanse of the world around it. It was designed for exploration and discovery, but this time, something unexpected caught its attention.

As it moved through the dense forest, the robot's sensors picked up faint sounds that seemed to come from somewhere in the distance. Curiosity piqued, it began to follow

=== temperature=0.7 ===
In the quiet hum of its own machinery, the small robot named Zephyr had been exploring the vast halls of the sprawling industrial complex that served as its home for many years. It had been programmed to perform various tasks such as cleaning and maintenance but had always found itself drawn to the mysterious corridors and rooms it could not access.

One day, while wandering through an abandoned section of the complex, Z

=== temperature=1.5 ===
In the depths of darkness, under layers of time and dust, a long-forgotten door stood silent but watchful

At `temperature=0.1` the model plays it safe with generic phrasing. At `0.7` the prose has more personality. At `1.5` the language becomes vivid and surprising — though coherence may occasionally suffer.

## 7  `top_k` — Vocabulary Truncation

In **Top-*k* sampling** (Fan et al., 2018), only the *k* most probable tokens at each step are kept; the rest are set to zero probability before sampling.

```
Full distribution:    [0.30,  0.20,  0.15,  0.10,  0.08,  0.07, …]
top_k=3  →  keep:     [0.30,  0.20,  0.15,     0,     0,     0, …]  → re-normalise → sample
```

| `top_k` | Candidate pool | Output character |
|---|---|---|
| 1 | 1 token (greedy) | Deterministic |
| 10–20 | Narrow | Focused, conservative |
| 50–100 | Wide | Varied, creative |

GPT-2 popularised `top_k=50` as a practical default.

In [24]:
for k in [1, 10, 50]:
    torch.manual_seed(42)
    print(f"=== top_k={k} ===")
    print(generate(CREATIVE_PROMPT, do_sample=True, max_new_tokens=80, top_k=k, temperature=0.9))
    print()

=== top_k=1 ===
Once upon a time, there was a small robot named Robby. He had been programmed to perform various tasks for his human masters, but he often found himself bored and lonely. One day, while exploring the vast warehouse where his creators kept their robots, Robby stumbled upon an old library.

The library was filled with books of all kinds, from science fiction novels to ancient texts in foreign languages.

=== top_k=10 ===
In the quiet hum of its own machinery, a small robot named Zephyr had been exploring the vast halls of the futuristic city it resided in. It had been tasked with mapping out the layout and identifying any potential hazards within the sprawling metropolis.

As Zephyr moved through the corridors, it encountered unexpected obstacles - some were physical barriers, others were invisible to the human eye but still presented

=== top_k=50 ===
In the quiet hum of its own machinery, a small robot named Zephyr had been exploring the vast halls of the world's larges

`top_k=1` produces the same greedy output every run. As *k* grows, sentence structure and vocabulary diversify — notice the difference in opening phrases and descriptive choices.

## 8  `top_p` — Nucleus Sampling

**Nucleus sampling** (Holtzman et al., 2020) selects the *smallest* set of tokens whose cumulative probability ≥ *p*, then samples from that set.

```
Sorted probabilities: [0.35, 0.25, 0.20, 0.10, 0.05, 0.03, 0.02, …]
top_p=0.80  →  keep until cumsum ≥ 0.80:  [0.35, 0.25, 0.20]  → re-normalise → sample
```

Unlike `top_k`, the **nucleus size adapts** at each step:
- Confident model (peaked distribution) → small nucleus (2–3 tokens)
- Uncertain model (flat distribution) → large nucleus (many tokens)

**Tip:** `top_p=0.9` with `top_k=0` is a popular nucleus-only configuration. The two filters can also be combined — the stricter one applies.

In [25]:
# Set top_k=0 to disable it and isolate the effect of top_p
for p in [0.5, 0.9, 1.0]:
    torch.manual_seed(42)
    print(f"=== top_p={p}  (top_k disabled) ===")
    print(generate(CREATIVE_PROMPT, do_sample=True, max_new_tokens=80,
                   top_p=p, top_k=0, temperature=0.9))
    print()

=== top_p=0.5  (top_k disabled) ===
In the quiet hum of its own machinery, the small robot named Zephyr had been exploring the vast halls of the futuristic city it called home. Its circuits buzzed with information and data, processing everything from sensor inputs to internal calculations. But one day, as it was wandering through corridors, something caught its attention.

The robot's sensors picked up faint echoes of conversation, whispers in a language that

=== top_p=0.9  (top_k disabled) ===
In the quiet hum of its own machinery, a small robot found itself in an unexpected place - an old library that had lain largely forgotten for decades. Its circuits buzzed with a mix of curiosity and concern as it surveyed the stacks of books, their spines so many against the grey walls.

The bookshelves were vast; they seemed to stretch endlessly into space, filled not just with volumes on

=== top_p=1.0  (top_k disabled) ===
In the quiet hum of its own machinery, a small robot found itself in 

At `top_p=0.5` the model is confined to its most confident choices, producing safe but predictable text. At `1.0` any token can be sampled, leading to more unusual — and occasionally less coherent — continuations.

## 9  `repetition_penalty` — Penalising Repeated Tokens

Before sampling, logits are **scaled down for any token that already appears** in the current sequence:

$$z'_i = \begin{cases} z_i / \text{penalty} & \text{if token } i \text{ has been generated before} \\ z_i & \text{otherwise} \end{cases}$$

| Value | Effect |
|---|---|
| `1.0` *(default)* | No penalty — logits unchanged |
| `> 1.0` (e.g. 1.1–1.3) | Discourages repetition — typical range |
| `< 1.0` | Encourages repetition (rarely useful) |

This operates at the **individual-token level**: even a single prior occurrence reduces that token's probability. It is most noticeable in long or descriptive generations where the model tends to reuse favoured words.

In [26]:
REPEAT_PROMPT = "Describe the ocean: its colour, its sounds, its movement, and its depth."

for penalty in [1.0, 1.2, 1.5]:
    torch.manual_seed(42)
    print(f"=== repetition_penalty={penalty} ===")
    print(generate(REPEAT_PROMPT, do_sample=True, max_new_tokens=100,
                   repetition_penalty=penalty, temperature=0.7, top_k=50))
    print()

=== repetition_penalty=1.0 ===
The ocean is a vast and vastly complex environment that is constantly in motion. It is a deep blue in colour, with a depth that can vary greatly depending on where you are in the ocean. The water is often murky and full of plankton and other tiny organisms, giving it a slightly greenish tint. 

The ocean is constantly in motion, with tides, currents, and waves that create a dynamic and ever-changing environment. The sound of the ocean can be quite loud, with the

=== repetition_penalty=1.2 ===
The ocean is an endless expanse of blue that stretches as far as one can see in all directions - it covers about 71% of Earth's surface area.

The color of the ocean varies depending on several factors such as light conditions (daytime or nighttime), water temperature, salinity, turbidity, chlorophyll concentration from phytoplankton blooms, distance away from shore where dissolved organic matter accumulates, underwater topography like kelp forests which create dark

At `1.0` the model may reuse the same descriptive words. At `1.2` repetitions drop noticeably. At `1.5` the model is forced to find alternative vocabulary at every step — inventive, but occasionally awkward.

## 10  `num_beams` — Beam Search

Instead of committing greedily to one token at each step, **beam search** keeps the *B* most probable partial sequences (beams) in parallel. At each step every beam is extended with all possible next tokens, and only the top-*B* combined sequences survive.

```
Step 1:  "The robot"  →  beam 1: "The robot carefully"  (log-prob −2.1)
                         beam 2: "The robot slowly"      (log-prob −2.4)
                         beam 3: "The robot gently"       (log-prob −2.6)
…each beam continues independently until EOS
```

| | Greedy (`num_beams=1`) | Beam (`num_beams=5`) |
|---|---|---|
| Speed | Fast | Slower (≈ ×B) |
| Output | Locally optimal | Globally more probable |
| Diversity | Low | Low |

> Beam search excels at tasks with a **single best answer** (translation, summarisation). For open-ended generation, sampling usually sounds more natural.

Beam search requires `do_sample=False`.

In [27]:
SUMMARY_PROMPT = "Summarise in two sentences the importance of sleep for cognitive performance."

print("=== Greedy  (num_beams=1) ===")
print(generate(SUMMARY_PROMPT, do_sample=False, num_beams=1, max_new_tokens=100))

print("\n=== Beam search  (num_beams=5) ===")
print(generate(SUMMARY_PROMPT, do_sample=False, num_beams=5,
               max_new_tokens=100, early_stopping=True))

=== Greedy  (num_beams=1) ===
Sleep is crucial for cognitive function, as it helps to consolidate memories and skills, improve mood, and enhance decision-making abilities. Without adequate sleep, cognitive performance can suffer, leading to difficulties with learning, problem-solving, and overall mental health.

=== Beam search  (num_beams=5) ===
Sleep is crucial for cognitive performance as it is essential for memory consolidation, learning, and overall brain function. Adequate sleep is necessary for optimal cognitive performance, including attention, concentration, and decision-making.


Both outputs are factually similar, but beam search tends to produce more fluent sentence structure because it can abandon locally attractive but globally sub-optimal word choices.

## 11  `no_repeat_ngram_size` — Hard N-gram Penalty

`no_repeat_ngram_size=n` sets the logit of any token that would **complete a previously-seen *n*-gram** to −∞, making it impossible to repeat any exact sequence of *n* consecutive tokens.

| Parameter | Mechanism | Granularity |
|---|---|---|
| `repetition_penalty` | Soft discount on any previously seen *individual* token | Single token |
| `no_repeat_ngram_size` | Hard ban on any repeated *n*-gram | Sequence of *n* tokens |

**Caution:** An aggressive setting (e.g. `n=2`) can prevent the model from repeating even natural phrases — for instance, a travel article about "New York" could only mention the city once.

In [28]:
for n in [0, 2, 4]:
    torch.manual_seed(42)
    label = "disabled" if n == 0 else f"ban any {n}-token repeat"
    print(f"=== no_repeat_ngram_size={n}  ({label}) ===")
    print(generate(REPEAT_PROMPT, do_sample=True, max_new_tokens=100,
                   no_repeat_ngram_size=n, temperature=0.7, top_k=50))
    print()

=== no_repeat_ngram_size=0  (disabled) ===
The ocean is an endless expanse of blue that stretches as far as the eye can see. Its surface is calm or rough depending on the time of day, but no matter how it looks, it never changes color. The water itself is clear, with a slight sheen to it due to the reflection of sunlight.

As you move deeper into the ocean, you will hear the sound of waves crashing against the shore. At first, this sound seems distant, like a low hum, but as you

=== no_repeat_ngram_size=2  (ban any 2-token repeat) ===
The ocean is an endless expanse of blue that stretches as far as the eye can see. Its surface is calm or rough depending on the time of day, but no matter how it looks, it never changes color. The water itself is clear, with a slight sheen to it due to the reflection of sunlight.

As you move deeper into the sea, the sound becomes louder. It's a constant hum, like waves crashing against each other. You can hear different types of marine life - fish

=== 

With `n=0` (disabled) short phrases may recur. At `n=2` no consecutive word pair appears twice, which sometimes forces awkward substitutions. At `n=4` natural language flows normally while still preventing structural repetition — a good default for longer outputs.

## Summary

| Parameter | What it controls | Practical default |
|---|---|---|
| `max_new_tokens` | Maximum output length | Task-dependent (50–500) |
| `do_sample` | Greedy vs. stochastic | `True` for creative, `False` for factual |
| `temperature` | Distribution sharpness | 0.6–0.9 for most tasks |
| `top_k` | Hard vocabulary size limit | 40–50 |
| `top_p` | Dynamic vocabulary size limit | 0.9–0.95 |
| `repetition_penalty` | Soft per-token repetition cost | 1.1–1.2 |
| `num_beams` | Beam search width | 1 (sampling) or 4–5 (beam) |
| `num_return_sequences` | Number of independent outputs | 1 (default), 3–5 (candidates) |
| `no_repeat_ngram_size` | Hard n-gram repetition ban | 3–4 for long-form text |

---

**Recipe for open-ended / creative generation:**
```python
generate(prompt, do_sample=True, temperature=0.8, top_p=0.9,
         repetition_penalty=1.1, max_new_tokens=200)
```

**Recipe for factual / structured output:**
```python
generate(prompt, do_sample=False, num_beams=4,
         no_repeat_ngram_size=3, early_stopping=True, max_new_tokens=150)
```

## References

* HuggingFace `.generate()` documentation — https://huggingface.co/docs/transformers/main_classes/text_generation
* How to generate text (HuggingFace blog) — https://huggingface.co/blog/how-to-generate
* Top-K sampling — Fan et al. (2018) — https://arxiv.org/abs/1805.04833
* Nucleus / Top-p sampling — Holtzman et al. (2020) — https://arxiv.org/abs/1904.09751
* QLoRA / BitsAndBytes quantisation — Dettmers et al. (2023) — https://arxiv.org/abs/2305.14314
* Qwen2.5 model — https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct